# Final Experiments

Runs the POSTERSUM OCR, spatial grouping, Qwen2.5-VL summarization, and ROUGE evaluation pipeline for the second test-split poster. The results files are created only after a successful run.

Run this notebook from the project root in Google Colab.

In [ ]:
!apt-get -qq update && apt-get -qq install -y tesseract-ocr
%pip install -q -r requirements.txt

In [ ]:
import csv
import sys
import tempfile
from pathlib import Path

from datasets import load_dataset
from PIL import Image, ImageDraw

ROOT = Path.cwd()
if not (ROOT / 'src' / 'final_pipeline.py').exists():
    ROOT = ROOT.parent
if not (ROOT / 'src' / 'final_pipeline.py').exists():
    raise FileNotFoundError('Run this notebook from the project root in Colab.')
sys.path.insert(0, str(ROOT))

from src.final_pipeline import (
    DATASET_ID,
    build_spatial_ocr,
    calculate_rouge,
    download_poster,
    filter_ocr,
    generate_summary,
    run_ocr,
)

SPLIT = 'test'
SAMPLE_INDEX = 1  # Second test-split item (poster 2)
RESULTS_DIR = ROOT / 'results'
CSV_OUTPUT = RESULTS_DIR / 'final_poster_results.csv'
IMAGE_OUTPUT = RESULTS_DIR / 'spatial_ocr_poster2_line_level.png'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if CSV_OUTPUT.exists() or IMAGE_OUTPUT.exists():
    raise FileExistsError('A results artifact already exists; move it before regenerating to avoid overwriting it.')

In [ ]:
dataset = load_dataset(DATASET_ID, split=SPLIT)
sample = dataset[SAMPLE_INDEX]

with tempfile.TemporaryDirectory(prefix='postersum-poster2-') as temporary_directory:
    poster_path = download_poster(
        sample['image_url'], Path(temporary_directory) / 'poster2.jpg'
    )
    raw_ocr = run_ocr(poster_path)
    filtered_ocr = filter_ocr(raw_ocr)
    spatial_ocr = build_spatial_ocr(filtered_ocr)
    summary = generate_summary(poster_path, spatial_ocr)

    poster = Image.open(poster_path).convert('RGB')
    draw = ImageDraw.Draw(poster)
    for line in spatial_ocr:
        box = line['bounding_box']
        draw.rectangle(
            (box['left'], box['top'], box['left'] + box['width'], box['top'] + box['height']),
            outline=(255, 80, 30),
            width=max(2, poster.width // 1000),
        )
    poster.save(IMAGE_OUTPUT)

rouge = calculate_rouge(summary, sample['abstract'])
result = {
    'dataset': DATASET_ID,
    'split': SPLIT,
    'sample_index': SAMPLE_INDEX,
    'title': sample.get('title', ''),
    'reference_abstract': sample['abstract'],
    'generated_summary': summary,
    'raw_ocr_words': len(raw_ocr),
    'filtered_ocr_words': len(filtered_ocr),
    'rouge1_f1': rouge['rouge1'],
    'rouge2_f1': rouge['rouge2'],
    'rougeL_f1': rouge['rougeL'],
}
with CSV_OUTPUT.open('w', encoding='utf-8', newline='') as output_file:
    writer = csv.DictWriter(output_file, fieldnames=result.keys())
    writer.writeheader()
    writer.writerow(result)

print('Title:', result['title'])
print('OCR words:', result['raw_ocr_words'])
print('Words at confidence >= 60:', result['filtered_ocr_words'])
print('ROUGE F1:', {key: result[key] for key in ('rouge1_f1', 'rouge2_f1', 'rougeL_f1')})
print('Saved:', CSV_OUTPUT)
print('Saved:', IMAGE_OUTPUT)